# Exercise 1: IEEE 802.15.4

Pietro Pizzoccheri 10797420 [team leader]  
Lorenzo Bardelli 10831941

## Data
- Poisson peaks per 10 s window: $\lambda = 0.4$
- Window duration: $T_w = 10$ s
- PHY bitrate: $R = 250$ kbps
- Packet size: $L = 128$ bytes
- Number of vibration nodes: 3
- Data sent per window:
  - 0 peaks -> 512 bytes
  - 1 peak -> 2 KB = 2048 bytes
  - more than 1 peak -> 4 KB = 4096 bytes

$1$ KB = $1024$ bytes.

## Q1 : Compute the Probability Mass Function of the output rate: 𝑃(𝑟=𝑟0),𝑃(𝑟=𝑟1),𝑃(𝑟=𝑟2)
where 𝑟0, 𝑟1, and 𝑟2 are the output rates when the node detects 0, 1, or more than 1
vibration peaks, respectively.

In [1]:
import math

lam = 0.4
Tw = 10.0
R = 250_000  # bit/s
L_bytes = 128
nodes = 3

# poisson probabilities
p0 = math.exp(-lam)         # probability of 0 packets in the buffer
p1 = lam * math.exp(-lam)   # probability of exactly 1 packet in the buffer
pgt1 = 1 - p0 - p1          # probability of more than 1 packet in the buffer

# rates (bit/s)
r0 = 512 * 8 / Tw
r1 = 2048 * 8 / Tw
r2 = 4096 * 8 / Tw

print('PMF of output rate:')
print("r0=" f'{r0:.1f} bit/s with P={p0:.4f}')
print("r1=" f'{r1:.1f} bit/s with P={p1:.4f}')
print("r2=" f'{r2:.1f} bit/s with P={pgt1:.4f}')
print()
print(f'P(r = r0)  = {p0:.4f}')
print(f'P(r = r1) = {p1:.4f}')
print(f'P(r = r2) = {pgt1:.4f}')


PMF of output rate:
r0=409.6 bit/s with P=0.6703
r1=1638.4 bit/s with P=0.2681
r2=3276.8 bit/s with P=0.0616

P(r = r0)  = 0.6703
P(r = r1) = 0.2681
P(r = r2) = 0.0616


## Q2 : Based on the output rate PMF, compute a consistent CFP slot assignment for the system.

In [ ]:
# packets needed in each case over one 10 s window
n0 = 512 // 128      # 4
n1 = 2048 // 128     # 16
n2 = 4096 // 128     # 32

# Worst-case CFP sizing 
L_bits = L_bytes * 8
slots_per_node = 8
BI = (slots_per_node * L_bits) / r2
beacon_intervals_per_window = Tw / BI
total_cfp_slots = slots_per_node * nodes

print(f'Worst-case packets per node per 10 s window = {n2}')
print(f'Beacon interval BI = {BI:.3f} s')
print(f'Beacon intervals per 10 s window = {beacon_intervals_per_window:.0f}')
print(f'CFP slots per node per BI = {slots_per_node}')
print(f'Total CFP slots per BI (3 nodes) = {total_cfp_slots}')


Worst-case packets per node per 10 s window = 32
Beacon interval BI = 2.500 s
Beacon intervals per 10 s window = 4
CFP slots per node per BI = 8
Total CFP slots per BI (3 nodes) = 24


## Q3: Compute:
- $T_s$
- $N_{slots}$ in the CFP
- $T_{active}$
- $T_{inactive}$
- The duty cycle of the system

In [3]:
# slot time, active time, inactive time, duty cycle
Ts = (L_bytes * 8) / R  # seconds
Tactive = (total_cfp_slots + 1) * Ts  # +1 beacon slot
Tinactive = BI - Tactive
DC = Tactive / BI



print(f'BI        = {BI:.3f} s')
print(f'Ts        = {Ts*1000:.3f} ms')
print(f'N_CFP     = {total_cfp_slots} slots')
print(f'Tactive   = {Tactive*1000:.3f} ms')
print(f'Tinactive = {Tinactive:.6f} s')
print(f'DutyCycle = {DC*100:.3f}%')

BI        = 2.500 s
Ts        = 4.096 ms
N_CFP     = 24 slots
Tactive   = 102.400 ms
Tinactive = 2.397600 s
DutyCycle = 4.096%


## Q4: How many additional vibration monitoring nodes can be added while keeping the duty cycle below 10%?

In [4]:
# how many nodes can we support with duty cycle < 10%?
Tactive_max = 0.10 * BI
N_active_slots_max = Tactive_max / Ts
N_cfp_max = N_active_slots_max - 1  # reserve 1 beacon slot
N_nodes_max = math.floor(N_cfp_max / slots_per_node)
N_additional = N_nodes_max - nodes

print(f'Max nodes with DC<10% = {N_nodes_max}')
print(f'Additional nodes      = {N_additional}')

Max nodes with DC<10% = 7
Additional nodes      = 4


## Final answers
- $P(r_0)=0.6703$, $P(r_1)=0.2681$, $P(r_2)=0.0616$
- $BI = 2.5$ s.
- CFP slots per node per beacon interval: **8**
- Total CFP slots per beacon interval for 3 nodes: **24**
- Slot time: **4.096 ms**
- Active time (with beacon slot): **102.400 ms**
- Inactive time: **2397.600 ms**
- Duty cycle: **4.096%**
- Additional nodes while keeping duty cycle below 10%: **4**
